# 목표

이 노트북은 Ray Train과 PyTorch를 사용하여 **NYC 택시 수요 시계열 데이터**를 활용해 Sequence-to-Sequence Transformer 모델을 분산 학습하는 방법을 실습하기 위한 파일입니다. 아래 각 셀의 지시사항에 따라 빈 셀(Code)을 추가하여 코드를 작성하세요.

**[Step 1: 라이브러리 설치 및 환경 설정]**

- 환경 변수 `RAY_TRAIN_V2_ENABLED`를 `"1"`로 설정하세요.
- 필요한 패키지를 설치하는 명령어를 작성하세요:
  설치할 패키지: `ray[train]`, `datasets`, `torch`, `pyarrow`, `pandas`, `matplotlib`, `ipywidgets`

**[Step 2: 모듈 임포트]**

- 기본 유틸리티: `os`, `math`, `shutil`
- 데이터 처리: `numpy`, `pandas`, `pyarrow.parquet` 등
- 시각화: `matplotlib.pyplot`
- PyTorch 관련: `torch` (nn, optim), `torch.utils.data.Dataset`, `DataLoader`
- Ray Train 관련: `ray`, `ray.data`, `ray.train` (ScalingConfig, RunConfig, Checkpoint 등), `ray.train.torch` (prepare_model, prepare_data_loader, TorchTrainer)

**[Step 3: 데이터 준비 및 Parquet 포맷으로 저장]**

다음 데이터 처리 과정을 구현하세요:
- 온라인에 공개된 NYC 택시 데이터(csv)를 다운로드하여 30분 간격으로 리샘플링(resampling)하고 데이터를 정규화(z-score normalization)하세요.
- 과거 1주일(168개 스텝)의 입력 윈도우(`past`)로 향후 24시간(48개 스텝)을 예측(`future`)하도록 데이터를 슬라이딩 윈도우(Sliding Window) 형태로 구성하세요.
- 시간 순서를 유지하며 학습(train)과 검증(val) 데이터로 분할한 뒤, `pyarrow`를 사용해 공유 스토리지 경로(예: `/tmp/nyc_taxi_ts/parquet`)에 Parquet 파일로 저장하세요.

**[Step 4: 데이터로더(DataLoader) 구축]**

- 저장된 Parquet 파일로부터 데이터를 읽어오는 PyTorch `Dataset` 상속 클래스(예: `TaxiWindowDataset`)를 작성하세요.
- 이 데이터셋을 바탕으로 DataLoader를 생성하고, Ray Train이 분산 환경에서 다룰 수 있도록 `prepare_data_loader()`로 감싸주는 `build_dataloader(parquet_path, batch_size, shuffle)` 함수를 구현하세요.

**[Step 5: 시계열 Transformer 모델 아키텍처 정의]**

- `nn.Module`을 상속받아 주기적 특징을 반영하는 `PositionalEncoding` 클래스를 구현하세요.
- `nn.Transformer` 모듈을 활용하여 입력(`past`)과 디코더 입력(`decoder_input`)을 받아 예측값(`future`)을 출력하는 `TimeSeriesTransformer` 클래스를 작성하세요.

**[Step 6: 분산 학습 함수 (train_loop_per_worker) 구현]**

- 인자로 `config`를 받는 `train_loop_per_worker` 함수를 작성하세요.
- Step 5에서 정의한 모델을 생성하고 `prepare_model`을 사용해 분산 처리를 준비하세요.
- `get_checkpoint()`를 확인해 이전에 저장된 체크포인트가 있으면 가중치를 불러와 학습을 재개(Resume)할 수 있도록 구현하세요.
- 에폭마다 모델을 학습(Teacher Forcing 적용)하고 검증(Validation)을 수행하세요.
- 0번 워커(`rank == 0`)에서만 체크포인트 디렉토리를 만들어 모델 가중치를 저장하고, `train.report`를 통해 메트릭 및 체크포인트를 보고하세요.

**[Step 7: 분산 학습 실행]**

- `TorchTrainer` 객체를 생성하세요.
- `train_loop_per_worker` 함수와 하이퍼파라미터(`train_loop_config`)를 지정하고, 워커 수(`num_workers`)가 포함된 `ScalingConfig`를 설정하세요.
- 장애 복구를 위한 `FailureConfig`와 체크포인트 유지 전략이 담긴 `RunConfig`를 포함시키세요.
- `ray.init(address="auto")`를 명시적으로 호출한 뒤, `trainer.fit()`을 실행하여 분산 학습을 시작하세요.

**[Step 8: 결과 시각화 및 Ray Data 인퍼런스]**

- 앞선 학습의 결과 객체(`result`)에서 `metrics_dataframe`을 추출하여 에폭별 학습 및 검증 손실(Loss) 곡선을 `matplotlib.pyplot`으로 시각화하세요.
- 최상의 체크포인트에서 모델을 단일 워커로 로드하는 인퍼런스용 클래스(예: `TimeSeriesBatchPredictor`)를 작성하세요.
- 최신 과거 데이터(`past`)를 사용해 `ray.data`의 `map_batches` 메서드로 미래 예측(Forecast)을 수행하고, 원래 스케일로 역정규화한 뒤 그래프로 출력해 보세요.